# Core

> Core helpers: path resolution, config loading, and shared utilities

In [ ]:
#| default_exp core

In [ ]:
#| export
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
from typing import Any

In [ ]:
#| export
def pkg_root() -> Path:
    "Return the root directory of the claude_plugins package."
    return Path(__file__).parent

In [ ]:
#| export
def find_project_root(start: Path | None = None) -> Path:
    """Walk up from `start` (default: cwd) until we find pyproject.toml, settings.ini, or .git."""
    p = Path(start or Path.cwd()).resolve()
    markers = {'pyproject.toml', 'settings.ini', '.git'}
    for parent in [p, *p.parents]:
        if any((parent / m).exists() for m in markers):
            return parent
    return p  # fallback: use start

In [ ]:
#| export
def claude_dir(project_root: Path | None = None) -> Path:
    "Return the .claude directory path, creating it if needed."
    root = project_root or find_project_root()
    d = root / '.claude'
    d.mkdir(exist_ok=True)
    return d

In [ ]:
#| export
def read_json(path: Path) -> dict:
    "Read a JSON file; return empty dict if missing."
    if path.exists():
        return json.loads(path.read_text())
    return {}

In [ ]:
#| export
def write_json(path: Path, data: dict, indent: int = 2) -> None:
    "Write a JSON file, creating parent directories as needed."
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=indent) + '\n')

In [ ]:
#| export
def merge_json(path: Path, new_data: dict) -> dict:
    """Deep-merge `new_data` into existing JSON at `path`. Lists are extended, not replaced."""
    existing = read_json(path)
    merged = _deep_merge(existing, new_data)
    write_json(path, merged)
    return merged


def _deep_merge(base: dict, override: dict) -> dict:
    result = dict(base)
    for k, v in override.items():
        if k in result and isinstance(result[k], dict) and isinstance(v, dict):
            result[k] = _deep_merge(result[k], v)
        elif k in result and isinstance(result[k], list) and isinstance(v, list):
            # Deduplicate by serialising; preserve order
            seen = {json.dumps(item, sort_keys=True) for item in result[k]}
            extras = [item for item in v if json.dumps(item, sort_keys=True) not in seen]
            result[k] = result[k] + extras
        else:
            result[k] = v
    return result

In [ ]:
#| export
HOOKS_SETTINGS = {
    "hooks": [
        {
            "event": "PreToolUse",
            "matcher": "Bash",
            "action": {
                "type": "command",
                "command": "uv run python -m claude_plugins.hooks.safecmd_hook"
            }
        },
        {
            "event": "PostToolUse",
            "matcher": "Edit|Write",
            "action": {
                "type": "command",
                "command": "uv run python -m claude_plugins.hooks.exhash_hook"
            }
        },
        {
            "event": "SessionStart",
            "action": {
                "type": "command",
                "command": "uv run python -m claude_plugins.hooks.index_hook"
            }
        }
    ]
}

MCP_CONFIG = {
    "mcpServers": {
        "exhash": {
            "type": "stdio",
            "command": "uv",
            "args": ["run", "python", "-m", "claude_plugins.mcp_exhash"]
        },
        "safecmd": {
            "type": "stdio",
            "command": "uv",
            "args": ["run", "python", "-m", "claude_plugins.mcp_safecmd"]
        },
        "safepyrun": {
            "type": "stdio",
            "command": "uv",
            "args": ["run", "python", "-m", "claude_plugins.mcp_safepyrun"]
        }
    }
}

## Tests

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    # Test merge_json creates file and deep-merges lists
    p = root / 'test.json'
    merge_json(p, {"hooks": [{"event": "A"}]})
    merge_json(p, {"hooks": [{"event": "B"}]})
    data = read_json(p)
    assert len(data['hooks']) == 2, f"Expected 2 hooks, got {data['hooks']}"
    # No-duplicate merging
    merge_json(p, {"hooks": [{"event": "A"}]})
    data = read_json(p)
    assert len(data['hooks']) == 2, "Should not add duplicate"
print('core tests passed')